
# Klike Data Science Challenge
## Parte 3 — Recommendations Engine

**Empresa:** Millai | Klike  
**Dataset:** `klike_challenge_dataset.csv`  
**Linguagem:** Python  
**Objetivo:** desenvolver um sistema de recomendações acionáveis e contextualizadas para otimização de criativos publicitários.

---

### Contextualização

Na Parte 1, foram investigadas as características das campanhas, a qualidade dos dados e as associações entre atributos dos criativos e métricas de desempenho.

Na Parte 2, foi desenvolvido um modelo de regressão Ridge reduzido, utilizando 10 características disponíveis antes da veiculação das campanhas para estimar o `klike_score`.

O modelo apresentou MAE de 6,685 e R² de 0,667 no conjunto de teste.

Nesta terceira etapa, o objetivo é transformar os resultados exploratórios e preditivos em um **Recommendations Engine**, capaz de analisar uma campanha e identificar oportunidades de melhoria.

O sistema utilizará uma abordagem híbrida, combinando:

- **Modelagem preditiva:** simulação de alterações nas características dos criativos por meio do Ridge reduzido.
- **Evidências estatísticas:** comparação de campanhas com características semelhantes, considerando seu contexto.
- **Priorização:** ordenação das sugestões conforme as diferenças previstas no Klike Score, respeitando a suficiência das evidências disponíveis.

As recomendações serão interpretadas como oportunidades de otimização, e não como garantias de melhoria. Diferenças preditivas e associações observacionais não constituem evidências de causalidade.



### Metodologia do Recommendations Engine

O desenvolvimento será organizado nas seguintes etapas:

1. **Preparação e reprodutibilidade:** carregamento do dataset e recuperação do Ridge reduzido desenvolvido na Parte 2.
2. **Validação das entradas:** identificação das características necessárias para analisar uma campanha e verificação de sua consistência.
3. **Geração de cenários alternativos:** definição de alterações acionáveis nas características dos criativos.
4. **Simulação preditiva:** utilização do Ridge para estimar diferenças no Klike Score entre a campanha original e os cenários alternativos.
5. **Análise contextual:** investigação das associações observadas entre características criativas e desempenho em campanhas comparáveis.
6. **Seleção e priorização:** filtragem de alterações não sustentadas pelas evidências e ordenação das oportunidades identificadas.
7. **Geração das recomendações:** apresentação de sugestões específicas, quantificadas e acompanhadas de suas limitações.
8. **Demonstração e validação:** execução do sistema em campanhas do dataset e verificação da consistência dos resultados.

**Princípios metodológicos:**

- Reutilizar o modelo selecionado na Parte 2, evitando treinamento desnecessário.
- Utilizar exclusivamente os registros de treinamento para construir as evidências estatísticas que fundamentam as recomendações.
- Preservar o conjunto de teste, evitando utilizá-lo na definição ou otimização das regras do engine.
- Não utilizar métricas pós-veiculação como entradas do modelo preditivo.
- Diferenciar explicitamente associações estatísticas, diferenças entre previsões e efeitos causais.
- Evitar recomendações baseadas em grupos com suporte amostral insuficiente.
- Não apresentar melhorias quantitativas sem uma estimativa calculada e devidamente contextualizada.

O sistema será desenvolvido inicialmente como um protótipo funcional em Python, utilizando funções independentes e reproduzíveis.



---

## 1. Preparação do ambiente e recuperação do modelo

### 1.1. Importação das bibliotecas e verificação dos arquivos

Nesta etapa, serão importadas as bibliotecas necessárias para manipulação dos dados, recuperação do modelo e reprodução da divisão entre treinamento e teste.

O Ridge reduzido foi salvo na Parte 2 utilizando `joblib`, juntamente com a lista ordenada das features utilizadas no treinamento.

Antes de carregar os arquivos, será verificada sua disponibilidade no diretório do projeto.

Essa abordagem permite reutilizar o pipeline previamente treinado sem realizar novos ajustes nos parâmetros do modelo.


In [1]:
from pathlib import Path
import sys
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

RANDOM_STATE = 42
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), Path.cwd().parent] if (p / 'src' / 'recommendations_engine.py').is_file()),
    None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Execute a partir da raiz do repositório ou da pasta notebooks/.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommendations_engine import RecommendationsEngine

DATA_PATH = PROJECT_ROOT / 'klike_challenge_dataset.csv'
MODEL_PATH = PROJECT_ROOT / 'models/ridge_klike.joblib'
FEATURES_PATH = PROJECT_ROOT / 'models/features_klike.joblib'
arquivos = {'Dataset original': DATA_PATH, 'Ridge reduzido': MODEL_PATH, 'Features do modelo': FEATURES_PATH}
status_arquivos = pd.DataFrame([
    {'Arquivo': nome, 'Caminho': str(caminho), 'Disponível': caminho.is_file()}
    for nome, caminho in arquivos.items()
])
display(status_arquivos)


,Arquivo,Caminho,Disponível
0,Dataset original,klike_challenge_dataset.csv,True
1,Ridge reduzido,models\ridge_klike.joblib,True
2,Features do modelo,models\features_klike.joblib,True



### 1.2. Carregamento dos dados e reconstrução da divisão treino/teste

O dataset será carregado sem alterações nos registros originais.

Em seguida, serão recuperados o pipeline Ridge treinado e a lista de features utilizada na Parte 2.

Para preservar a reprodutibilidade, será reconstruída a mesma divisão aleatória entre treinamento e teste, utilizando `test_size=0.20` e `random_state=42`.

As variáveis booleanas serão convertidas para a representação numérica adotada anteriormente, preservando os valores ausentes para tratamento pelo próprio pipeline.

A reconstrução da divisão é necessária porque as estatísticas contextuais do Recommendations Engine serão calculadas exclusivamente sobre as campanhas de treinamento.

O conjunto de teste permanecerá separado e não será utilizado na elaboração das recomendações.


In [2]:
engine = RecommendationsEngine(PROJECT_ROOT)
df = engine.df
modelo_ridge = engine.modelo_ridge
features_reduzidas = engine.features_reduzidas
X_train, X_test = engine.X_train, engine.X_test
y_train, y_test = engine.y_train, engine.y_test
df_train = engine.df_train

print('Modelo:', type(modelo_ridge.named_steps['model']).__name__)
print('Alpha:', modelo_ridge.named_steps['model'].alpha)
print('Etapas do pipeline:', list(modelo_ridge.named_steps.keys()))
print('Dataset original:', df.shape)
print('Treinamento:', X_train.shape)
print('Teste:', X_test.shape)
print('Features utilizadas:', len(features_reduzidas))
print('Interseção treino/teste:', len(set(X_train.index) & set(X_test.index)))


Modelo: Ridge
Alpha: 1.0
Etapas do pipeline: ['preprocessor', 'model']

Dimensões:
Dataset original: (500, 26)
Treinamento: (400, 10)
Teste: (100, 10)
Features utilizadas: 10

Features do modelo:
 1. platform
 2. category
 3. is_retargeting
 4. video_duration_s
 5. format
 6. has_subtitle
 7. has_cta
 8. has_hook
 9. has_face
10. text_density

Validação dos índices:
Interseção treino/teste: 0
Registros para evidência contextual: 400



---

## 2. Validação das entradas e definição das características modificáveis

### 2.1. Classificação das features

O Recommendations Engine utilizará as mesmas 10 features do Ridge reduzido, divididas em dois grupos:

**Características de contexto:** descrevem o ambiente e o direcionamento da campanha. Serão mantidas constantes durante as simulações para que as recomendações sejam contextualizadas.

**Características modificáveis:** representam aspectos do criativo que podem ser alterados para produzir cenários alternativos.

A classificação é importante para impedir que o sistema recomende alterações em características que não correspondem diretamente à otimização do criativo.

Nesta etapa, também serão verificados os valores categóricos presentes no conjunto de treinamento e a quantidade de registros ausentes.

As categorias conhecidas serão utilizadas para validar os inputs. Essa validação evita que valores desconhecidos sejam silenciosamente codificados pelo pipeline, comprometendo a interpretação das previsões.


In [3]:

# Características que representam o contexto da campanha
features_contexto = engine.features_contexto

# Características que podem ser modificadas no criativo
features_modificaveis = engine.features_modificaveis

# Classificação por tipo de dado
features_categoricas = engine.features_categoricas

features_booleanas = engine.features_booleanas

features_numericas = [
    "video_duration_s"
]

# Verificar se todas as features foram classificadas
assert set(features_contexto).isdisjoint(
    features_modificaveis
)

assert set(features_contexto + features_modificaveis) == set(
    features_reduzidas
)

assert set(
    features_categoricas + features_booleanas + features_numericas
) == set(features_reduzidas)

# Resumo das características
resumo_features = pd.DataFrame({
    "Feature": features_reduzidas,
    "Papel": [
        "Contexto" if feature in features_contexto
        else "Modificável"
        for feature in features_reduzidas
    ],
    "Tipo": [
        "Categórica" if feature in features_categoricas
        else "Booleana" if feature in features_booleanas
        else "Numérica"
        for feature in features_reduzidas
    ],
    "Valores ausentes no treino": [
        df_train[feature].isna().sum()
        for feature in features_reduzidas
    ]
})

display(resumo_features)

# Categorias efetivamente observadas no treinamento
categorias_validas = engine.categorias_validas

print("\nCategorias disponíveis no treinamento:")

for feature, categorias in categorias_validas.items():
    print(f"{feature}: {categorias}")

print("\nDistribuição da duração do vídeo no treinamento:")
display(
    df_train["video_duration_s"]
    .describe()
    .to_frame(name="video_duration_s")
)


,Feature,Papel,Tipo,Valores ausentes no treino
0,platform,Contexto,Categórica,0
1,category,Contexto,Categórica,0
2,is_retargeting,Contexto,Booleana,0
3,video_duration_s,Modificável,Numérica,34
4,format,Modificável,Categórica,0
5,has_subtitle,Modificável,Booleana,38
6,has_cta,Modificável,Booleana,0
7,has_hook,Modificável,Booleana,0
8,has_face,Modificável,Booleana,0
9,text_density,Modificável,Categórica,0



Categorias disponíveis no treinamento:
platform: ['LinkedIn', 'Meta', 'TikTok']
category: ['App Install', 'Branding', 'E-commerce', 'Lead Gen', 'SaaS']
format: ['horizontal', 'quadrado', 'vertical']
text_density: ['high', 'low', 'medium']

Distribuição da duração do vídeo no treinamento:


,video_duration_s
count,366.000000
mean,19.702186
std,14.168704
min,5.000000
25%,11.000000
50%,16.000000
75%,26.000000
max,115.000000



### 2.2. Preparação e validação de uma campanha individual

O protótipo receberá uma campanha representada por um dicionário Python ou por uma linha do dataset (`pandas.Series`).

A função de preparação terá as seguintes responsabilidades:

- Verificar se as 10 features necessárias estão presentes.
- Validar os valores categóricos utilizando apenas as categorias observadas no treinamento.
- Converter características booleanas para a representação numérica utilizada no Ridge.
- Verificar se a duração do vídeo é numericamente válida.
- Preservar valores ausentes nas características que já possuíam ausências no dataset e foram tratadas pelo pipeline.
- Organizar as características na ordem esperada pelo modelo.

O processo não utilizará `klike_score`, CTR, ROAS ou outras métricas pós-veiculação como entradas preditivas.

A função desenvolvida nesta seção servirá como interface de entrada para o Recommendations Engine, permitindo analisar campanhas existentes ou novas campanhas informadas manualmente.


In [4]:

import numpy as np

preparar_campanha = engine.preparar_campanha  # Fonte única: src/recommendations_engine.py
# Selecionar uma campanha do conjunto de treinamento
campanha_exemplo = df_train.iloc[0]

# Preparar o input
entrada_validada = preparar_campanha(campanha_exemplo)

print("Campanha selecionada:")
print(campanha_exemplo["campaign_id"])

print("\nInput validado para o Recommendations Engine:")
display(entrada_validada)


Campanha selecionada:
KLK-0250

Input validado para o Recommendations Engine:


,platform,category,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density
0,LinkedIn,App Install,0,22.0,vertical,1,1,1,0,medium



---

## 3. Geração de cenários alternativos

### 3.1. Definição das alterações candidatas

O Recommendations Engine avaliará possíveis modificações nas características dos criativos, preservando o contexto original de cada campanha.

Cada cenário será gerado alterando **uma única feature por vez**, permitindo comparar a previsão da campanha original com a previsão de uma configuração alternativa.

As modificações serão definidas conforme o tipo de variável:

- **Booleanas:** testar a inclusão de características ausentes, como hook, CTA, legendas e rosto humano.
- **Categóricas:** testar formatos e densidades de texto existentes no conjunto de treinamento.
- **Numéricas:** testar durações alternativas baseadas nos quartis observados em campanhas da mesma plataforma, sem assumir que esses valores sejam ideais.

As características `platform`, `category` e `is_retargeting` permanecerão constantes durante as simulações.

Valores ausentes não serão interpretados automaticamente como ausência de uma característica. Nesses casos, não serão geradas alterações para a feature até que seu valor original seja conhecido.

Os cenários representam **hipóteses de melhoria**, não recomendações finais. Sua seleção dependerá posteriormente das diferenças previstas pelo Ridge e das evidências estatísticas contextuais.

Para evitar data leakage, as alternativas baseadas na distribuição dos dados serão calculadas exclusivamente a partir do conjunto de treinamento.


In [5]:

gerar_cenarios = engine.gerar_cenarios  # Fonte única: src/recommendations_engine.py
# Gerar cenários para a campanha de exemplo
cenarios_exemplo = gerar_cenarios(campanha_exemplo)

print("Campanha:", campanha_exemplo["campaign_id"])
print("Total de cenários:", len(cenarios_exemplo))

print("\nAlterações candidatas:")
display(
    cenarios_exemplo[
        [
            "feature_alterada",
            "valor_original",
            "valor_proposto",
            "origem"
        ]
    ]
)

print("\nQuantidade de cenários por feature:")
display(
    cenarios_exemplo["feature_alterada"]
    .value_counts()
    .rename_axis("Feature")
    .reset_index(name="Cenários")
)


Campanha: KLK-0250
Total de cenários: 8

Alterações candidatas:


,feature_alterada,valor_original,valor_proposto,origem
0,has_face,0,1,Inclusão de característica ausente
1,format,vertical,horizontal,Categoria observada no treinamento
2,format,vertical,quadrado,Categoria observada no treinamento
3,text_density,medium,high,Categoria observada no treinamento
4,text_density,medium,low,Categoria observada no treinamento
5,video_duration_s,22.0,18,Quartis da duração por plataforma no treino
6,video_duration_s,22.0,26,Quartis da duração por plataforma no treino
7,video_duration_s,22.0,31,Quartis da duração por plataforma no treino



Quantidade de cenários por feature:


,Feature,Cenários
0,video_duration_s,3
1,format,2
2,text_density,2
3,has_face,1


In [6]:

valores_iguais = engine.valores_iguais  # Fonte única: src/recommendations_engine.py
validar_cenarios = engine.validar_cenarios  # Fonte única: src/recommendations_engine.py
validar_cenarios(
    campanha_exemplo,
    cenarios_exemplo
)


Todos os cenários passaram na validação.
Contexto preservado e apenas uma alteração por cenário.



---

## 4. Simulação preditiva com o Ridge reduzido

### 4.1. Estimativa das diferenças previstas

Após a geração dos cenários alternativos, será utilizado o Ridge reduzido desenvolvido na Parte 2 para estimar o Klike Score de cada configuração.

Para cada campanha, o procedimento será:

1. Calcular o Klike Score previsto para a configuração original.
2. Calcular o Klike Score previsto para cada cenário alternativo.
3. Obter a diferença entre a previsão alternativa e a previsão original.
4. Organizar os resultados para posterior avaliação contextual.

A diferença prevista será calculada por:

\[
\Delta_{\text{score}} = \hat{y}_{\text{alternativo}} - \hat{y}_{\text{original}}
\]

Valores positivos representam cenários com maior Klike Score previsto pelo modelo, enquanto valores negativos representam cenários com previsão inferior.

### Limitações

As diferenças estimadas representam comparações entre previsões de um modelo estatístico, não efeitos causais comprovados.

Além disso, o Ridge é um modelo linear aditivo, sem interações explícitas entre características criativas e plataformas. Portanto, diferenças específicas por contexto serão investigadas separadamente na análise estatística.

O modelo não será ajustado novamente nesta etapa. Serão utilizados somente os artefatos previamente treinados.

As previsões serão mantidas em sua escala original, sem truncamento automático, para preservar as diferenças estimadas e permitir identificar eventuais valores fora da faixa de 0 a 100.

A seleção definitiva das recomendações dependerá também da consistência das associações observadas no conjunto de treinamento.


In [9]:

simular_impactos = engine.simular_impactos  # Fonte única: src/recommendations_engine.py

In [10]:

# Simular a campanha utilizada nas etapas anteriores
score_original, simulacoes = simular_impactos(
    campanha_exemplo
)

print("Campanha:", campanha_exemplo["campaign_id"])
print(f"Klike Score previsto original: {score_original:.3f}")

print("\nQuantidade de cenários avaliados:", len(simulacoes))

print("\nResultados da simulação preditiva:")

display(
    simulacoes.round({
        "score_previsto_original": 3,
        "score_previsto_cenario": 3,
        "delta_score_previsto": 3
    })
)

# Verificações de consistência
assert np.isfinite(score_original)

assert np.isfinite(
    simulacoes[
        [
            "score_previsto_original",
            "score_previsto_cenario",
            "delta_score_previsto"
        ]
    ].to_numpy(dtype=float)
).all()

assert np.allclose(
    simulacoes["delta_score_previsto"],
    simulacoes["score_previsto_cenario"]
    - simulacoes["score_previsto_original"]
)

# Identificar eventuais previsões fora da escala
previsoes_fora_faixa = (
    ~simulacoes["score_previsto_cenario"].between(0, 100)
).sum()

print("\nVerificações:")
print("Diferenças calculadas corretamente.")
print("Previsões alternativas fora de 0 a 100:", previsoes_fora_faixa)
print(
    "Previsão original fora de 0 a 100:",
    not 0 <= score_original <= 100
)


Campanha: KLK-0250
Klike Score previsto original: 66.506

Quantidade de cenários avaliados: 8

Resultados da simulação preditiva:


,feature_alterada,valor_original,valor_proposto,origem,score_previsto_original,score_previsto_cenario,delta_score_previsto
0,has_face,0,1,Inclusão de característica ausente,66.506,78.256,11.750
1,video_duration_s,22.0,31,Quartis da duração por plataforma no treino,66.506,66.584,0.078
2,video_duration_s,22.0,26,Quartis da duração por plataforma no treino,66.506,66.541,0.035
3,video_duration_s,22.0,18,Quartis da duração por plataforma no treino,66.506,66.471,-0.035
4,format,vertical,quadrado,Categoria observada no treinamento,66.506,65.179,-1.326
5,format,vertical,horizontal,Categoria observada no treinamento,66.506,64.969,-1.537
6,text_density,medium,low,Categoria observada no treinamento,66.506,59.922,-6.584
7,text_density,medium,high,Categoria observada no treinamento,66.506,52.471,-14.035



Verificações:
Diferenças calculadas corretamente.
Previsões alternativas fora de 0 a 100: 0
Previsão original fora de 0 a 100: False



---

## 5. Evidências estatísticas contextuais

### 5.1. Preparação da base de referência

O Recommendations Engine híbrido combinará duas fontes complementares de informação:

- **Previsões do Ridge reduzido:** diferenças estimadas no Klike Score entre configurações alternativas.
- **Associações observacionais:** diferenças de desempenho entre grupos de campanhas que compartilham características de contexto.

As análises contextuais utilizarão exclusivamente os 400 registros de treinamento, evitando utilizar o conjunto de teste na construção das recomendações.

Serão consideradas inicialmente três métricas:

- `klike_score`: pontuação de qualidade do criativo.
- `ctr`: proporção de impressões que resultaram em cliques.
- `engagement_rate`: proporção de interações em relação às impressões.

Na Parte 1, foram identificadas inconsistências matemáticas no CTR. Portanto, será reproduzida a regra de tratamento utilizada na EDA: divergências superiores a 0,0001 entre o CTR registrado e o CTR recalculado serão desconsideradas nas análises dessa métrica.

Essa exclusão será aplicada apenas ao CTR, preservando as demais informações da campanha.

As comparações serão descritivas e não constituirão estimativas de efeitos causais.


In [11]:
df_evidencias = engine.df_evidencias.copy()
ctr_calculado = df_evidencias['clicks'] / df_evidencias['impressions'].replace(0, np.nan)
ctr_comparavel = df_evidencias['ctr'].notna() & ctr_calculado.notna()
ctr_inconsistente = ctr_comparavel & (df_evidencias['ctr'] - ctr_calculado).abs().gt(0.0001)
resumo_metricas = pd.DataFrame([
    {'Métrica': 'Klike Score', 'Registros disponíveis': df_evidencias['klike_score'].notna().sum()},
    {'Métrica': 'CTR consistente', 'Registros disponíveis': df_evidencias['ctr_consistente'].notna().sum()},
    {'Métrica': 'Engagement Rate', 'Registros disponíveis': df_evidencias['engagement_rate'].notna().sum()},
])
print(f'Registros no treinamento: {len(df_evidencias)}')
print(f'CTR comparáveis: {ctr_comparavel.sum()}')
print(f'CTR inconsistentes: {ctr_inconsistente.sum()}')
display(resumo_metricas)
assert set(df_evidencias.index).isdisjoint(set(X_test.index))


Preparação das evidências contextuais
Registros no treinamento: 400
CTR comparáveis: 400
CTR inconsistentes: 49

Disponibilidade das métricas:


,Métrica,Registros disponíveis
0,Klike Score,400
1,CTR consistente,351
2,Engagement Rate,375



Verificações de isolamento concluídas.



### 5.2. Distribuição das campanhas por contexto

Antes de calcular associações, será avaliada a quantidade de campanhas disponíveis em cada combinação de plataforma e categoria.

Esse diagnóstico é importante porque a segmentação excessiva pode produzir comparações baseadas em poucas observações, aumentando a instabilidade das estimativas.

A arquitetura proposta seguirá uma hierarquia de contextualização:

1. **Plataforma e categoria:** comparação mais específica, quando houver suporte amostral suficiente nos dois grupos.
2. **Plataforma:** comparação menos específica, utilizada quando a combinação anterior não possuir observações suficientes.
3. **Evidência insuficiente:** não será apresentada uma associação quantitativa contextual quando nenhum dos níveis atender aos critérios mínimos.

A primeira referência para suporte amostral será de pelo menos 10 observações válidas por grupo, seguindo o critério exploratório utilizado na Parte 1.

Essa regra é uma salvaguarda pragmática, não uma garantia de precisão estatística. Outras limitações, como variabilidade, confundimento e múltiplas comparações, também deverão ser consideradas.


In [12]:

print("Distribuição por plataforma:")

display(
    df_evidencias["platform"]
    .value_counts()
    .rename_axis("Plataforma")
    .reset_index(name="Campanhas")
)

print("\nDistribuição por plataforma e categoria:")

distribuicao_contexto = pd.crosstab(
    df_evidencias["platform"],
    df_evidencias["category"]
)

display(distribuicao_contexto)

print("\nCampanhas disponíveis no contexto do exemplo:")

contexto_exemplo = df_evidencias.loc[
    (df_evidencias["platform"] == campanha_exemplo["platform"])
    & (df_evidencias["category"] == campanha_exemplo["category"])
]

print(
    f"Plataforma: {campanha_exemplo['platform']}"
)
print(
    f"Categoria: {campanha_exemplo['category']}"
)
print(
    f"Campanhas no contexto: {len(contexto_exemplo)}"
)


Distribuição por plataforma:


,Plataforma,Campanhas
0,Meta,188
1,TikTok,142
2,LinkedIn,70



Distribuição por plataforma e categoria:


category,App Install,Branding,E-commerce,Lead Gen,SaaS
platform,,,,,
LinkedIn,15,8,24,12,11
Meta,32,22,61,36,37
TikTok,20,17,46,32,27



Campanhas disponíveis no contexto do exemplo:
Plataforma: LinkedIn
Categoria: App Install
Campanhas no contexto: 15



### 5.3. Comparação de desempenho entre configurações criativas

Para complementar as simulações do Ridge, serão calculadas diferenças observacionais entre campanhas com características criativas distintas.

Cada comparação considerará a configuração atual e uma configuração alternativa, utilizando uma hierarquia de contexto:

1. **Plataforma + categoria:** prioridade para campanhas do mesmo ambiente e segmento.
2. **Plataforma:** alternativa quando o agrupamento anterior possuir dados insuficientes.
3. **Evidência insuficiente:** resultado apresentado quando não houver suporte amostral adequado.

Será exigido um mínimo inicial de 10 observações válidas por grupo, para cada métrica analisada.

As comparações utilizarão médias dos grupos, apresentando:

- Quantidade de observações em cada grupo.
- Média observada na configuração original.
- Média observada na configuração alternativa.
- Diferença absoluta entre as médias.
- Diferença percentual relativa, quando aplicável.

Para o Klike Score, a diferença absoluta será expressa em pontos. Para CTR e taxa de engajamento, será expressa em pontos percentuais, acompanhada da variação percentual relativa.

**Importante:** diferenças observadas entre campanhas não representam o impacto causal de modificar um criativo. Os grupos podem diferir em outras características relevantes.

As comparações serão realizadas exclusivamente com os dados de treinamento. As variáveis numéricas, como a duração do vídeo, serão tratadas separadamente, pois exigem uma estratégia de agrupamento apropriada.


In [13]:

comparar_cenario_contextual = engine.comparar_cenario_contextual  # Fonte única: src/recommendations_engine.py

In [14]:

# Comparação contextual da principal alteração candidata
evidencias_rosto = comparar_cenario_contextual(
    campanha=campanha_exemplo,
    feature="has_face",
    valor_proposto=1,
    minimo_por_grupo=10
)

print("Campanha:", campanha_exemplo["campaign_id"])
print("Plataforma:", campanha_exemplo["platform"])
print("Categoria:", campanha_exemplo["category"])

print("\nAlteração avaliada:")
print("Adicionar rosto humano ao criativo")

print("\nEvidências estatísticas contextuais:")

display(
    evidencias_rosto.round({
        "Média original": 3,
        "Média proposta": 3,
        "Diferença observada": 3,
        "Variação relativa (%)": 2
    })
)

print("\nVerificação do suporte amostral:")

for _, resultado in evidencias_rosto.iterrows():

    print(
        f"{resultado['Métrica']}: "
        f"{resultado['Status']} | "
        f"{resultado['Contexto']} | "
        f"N={resultado['N original']} vs "
        f"N={resultado['N proposto']}"
    )


Campanha: KLK-0250
Plataforma: LinkedIn
Categoria: App Install

Alteração avaliada:
Adicionar rosto humano ao criativo

Evidências estatísticas contextuais:


,Métrica,Contexto,N original,N proposto,Média original,Média proposta,Diferença observada,Unidade,Variação relativa (%),Status
0,Klike Score,Plataforma,30,40,48.733,62.295,13.562,pontos,NaN,Comparação disponível
1,CTR consistente,Plataforma,27,35,6.057,8.626,2.570,p.p.,42.43,Comparação disponível
2,Engagement Rate,Plataforma,29,37,1.822,2.457,0.635,p.p.,34.87,Comparação disponível



Verificação do suporte amostral:
Klike Score: Comparação disponível | Plataforma | N=30 vs N=40
CTR consistente: Comparação disponível | Plataforma | N=27 vs N=35
Engagement Rate: Comparação disponível | Plataforma | N=29 vs N=37



### 5.4. Integração entre previsões e evidências contextuais

Nesta etapa, as simulações preditivas serão associadas às comparações estatísticas contextuais.

Para cada cenário categórico, serão reunidas:

- A característica modificada e sua configuração alternativa.
- A diferença prevista pelo Ridge reduzido.
- A diferença média observada no Klike Score.
- As diferenças observadas no CTR e na taxa de engajamento.
- O contexto e o suporte amostral de cada comparação.

Os resultados serão organizados em duas estruturas:

1. **Tabela consolidada:** uma linha por cenário, contendo as diferenças preditivas e observacionais disponíveis.
2. **Tabela detalhada:** uma linha por cenário e métrica, preservando o contexto, os tamanhos amostrais e o status da comparação.

As alterações de duração serão mantidas na tabela consolidada, mas suas evidências contextuais permanecerão pendentes até a definição de uma metodologia apropriada para variáveis numéricas.

Nesta etapa, não serão atribuídos pesos às métricas nem combinadas suas diferenças em um único indicador.

A integração permitirá investigar posteriormente a concordância entre o Ridge e os padrões observados, sem confundir previsão com causalidade.


In [16]:

integrar_evidencias_cenarios = engine.integrar_evidencias_cenarios  # Fonte única: src/recommendations_engine.py

In [17]:

# Integrar os cenários da campanha KLK-0250
resumo_integrado, detalhes_integrados = (
    integrar_evidencias_cenarios(
        campanha=campanha_exemplo,
        simulacoes=simulacoes,
        minimo_por_grupo=10
    )
)

print("Campanha:", campanha_exemplo["campaign_id"])
print("Total de cenários:", len(resumo_integrado))

print("\nResumo integrado:")
display(resumo_integrado.round(3))

print("\nDisponibilidade das evidências:")
display(
    resumo_integrado["Status evidencias"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Cenários")
)

# Exibir os detalhes da alteração has_face
cenario_rosto = resumo_integrado.loc[
    resumo_integrado["Feature"] == "has_face",
    "Cenario"
].iloc[0]

print("\nEvidências detalhadas — has_face:")
display(
    detalhes_integrados.loc[
        detalhes_integrados["Cenario"] == cenario_rosto
    ].round(3)
)

# Verificações de consistência
assert len(resumo_integrado) == len(simulacoes)

n_cenarios_categoricos = (
    simulacoes["feature_alterada"] != "video_duration_s"
).sum()

assert len(detalhes_integrados) == (
    n_cenarios_categoricos * 3
)

assert resumo_integrado["Cenario"].is_unique

assert np.allclose(
    resumo_integrado["Delta Ridge (pontos)"],
    simulacoes["delta_score_previsto"]
)

print("\nIntegração concluída com sucesso.")


Campanha: KLK-0250
Total de cenários: 8

Resumo integrado:


,Cenario,Feature,Original,Proposto,Delta Ridge (pontos),Status evidencias,Delta Klike observado (pontos),Delta CTR observado (p.p.),Delta Engagement observado (p.p.)
0,1,has_face,0,1,11.750,Completa,13.562,2.570,0.635
1,2,video_duration_s,22.0,31,0.078,Pendente: duracao numerica,NaN,NaN,NaN
2,3,video_duration_s,22.0,26,0.035,Pendente: duracao numerica,NaN,NaN,NaN
3,4,video_duration_s,22.0,18,-0.035,Pendente: duracao numerica,NaN,NaN,NaN
4,5,format,vertical,quadrado,-1.326,Parcial,14.089,NaN,-0.151
5,6,format,vertical,horizontal,-1.537,Parcial,16.459,NaN,-0.140
6,7,text_density,medium,low,-6.584,Completa,-5.090,0.909,-0.354
7,8,text_density,medium,high,-14.035,Completa,-15.875,-0.563,-0.162



Disponibilidade das evidências:


,Status,Cenários
0,Completa,3
1,Pendente: duracao numerica,3
2,Parcial,2



Evidências detalhadas — has_face:


,Cenario,Feature,Proposto,Metrica,Contexto,N original,N proposto,Diferenca observada,Unidade,Variacao relativa (%),Status
0,1,has_face,1,Klike Score,Plataforma,30,40,13.562,pontos,NaN,Comparação disponível
1,1,has_face,1,CTR consistente,Plataforma,27,35,2.570,p.p.,42.426,Comparação disponível
2,1,has_face,1,Engagement Rate,Plataforma,29,37,0.635,p.p.,34.874,Comparação disponível



Integração concluída com sucesso.



### 5.5. Análise contextual da duração do vídeo

A duração do vídeo é uma variável numérica contínua. Diferentemente das características booleanas e categóricas, comparações entre valores exatos podem produzir grupos pequenos e estimativas instáveis.

Para investigar associações entre duração e desempenho, os vídeos serão organizados em quatro faixas, definidas pelos quartis observados no conjunto de treinamento de cada plataforma.

Essa estratégia permite comparar campanhas de duração semelhante sem estabelecer arbitrariamente um tempo ideal.

Para cada cenário:

1. Identificar a faixa de duração da campanha original.
2. Identificar a faixa correspondente à duração proposta.
3. Comparar as métricas observadas entre essas faixas.
4. Priorizar o contexto plataforma + categoria quando houver pelo menos 10 observações válidas por grupo.
5. Utilizar o contexto da plataforma quando o grupo mais específico for insuficiente.

Quando as durações original e proposta pertencerem à mesma faixa, não será atribuída uma diferença observacional à alteração.

Os limites das faixas serão definidos exclusivamente com os dados de treinamento, evitando data leakage.

**Limitação:** os resultados representarão diferenças entre faixas de duração, não o impacto causal ou exato de modificar um vídeo de uma duração específica para outra.


In [18]:

comparar_duracao_contextual = engine.comparar_duracao_contextual  # Fonte única: src/recommendations_engine.py

In [19]:

# Cenários de duração da campanha de exemplo
cenarios_duracao = simulacoes.loc[
    simulacoes["feature_alterada"] == "video_duration_s"
].copy()

# Identificar os quartis de duração da plataforma
duracoes_referencia = df_evidencias.loc[
    df_evidencias["platform"] == campanha_exemplo["platform"],
    "video_duration_s"
].dropna()

print("Campanha:", campanha_exemplo["campaign_id"])
print("Plataforma:", campanha_exemplo["platform"])

print("\nQuartis da duração no treinamento:")
display(
    duracoes_referencia.quantile(
        [0.25, 0.50, 0.75]
    ).to_frame(name="Duração (s)")
)

# Comparar cada duração alternativa
resultados_duracao = []

for _, cenario in cenarios_duracao.iterrows():

    comparacao = comparar_duracao_contextual(
        campanha=campanha_exemplo,
        duracao_proposta=cenario["valor_proposto"],
        minimo_por_grupo=10
    )

    comparacao.insert(
        0,
        "Duração proposta (s)",
        float(cenario["valor_proposto"])
    )

    comparacao.insert(
        1,
        "Delta Ridge (pontos)",
        float(cenario["delta_score_previsto"])
    )

    resultados_duracao.append(comparacao)

evidencias_duracao = pd.concat(
    resultados_duracao,
    ignore_index=True
)

print("\nComparações contextuais da duração:")
display(evidencias_duracao.round(3))

print("\nStatus das comparações:")
display(
    evidencias_duracao["Status"]
    .value_counts()
    .rename_axis("Status")
    .reset_index(name="Comparações")
)

# Verificações
assert len(evidencias_duracao) == (
    len(cenarios_duracao) * 3
)

assert set(evidencias_duracao["Métrica"]) == {
    "Klike Score",
    "CTR consistente",
    "Engagement Rate"
}

print("\nAnálise contextual de duração concluída.")


Campanha: KLK-0250
Plataforma: LinkedIn

Quartis da duração no treinamento:


,Duração (s)
0.25,18.50
0.50,25.50
0.75,31.25



Comparações contextuais da duração:


,Duração proposta (s),Delta Ridge (pontos),Métrica,Contexto,Faixa original,Faixa proposta,N original,N proposto,Diferença observada,Unidade,Variação relativa (%),Status
0,31.0,0.078,Klike Score,Plataforma,2,3,16,16,4.600,pontos,NaN,Comparação disponível
1,31.0,0.078,CTR consistente,Plataforma,2,3,14,14,1.893,p.p.,25.161,Comparação disponível
2,31.0,0.078,Engagement Rate,Plataforma,2,3,14,16,0.235,p.p.,10.108,Comparação disponível
3,26.0,0.035,Klike Score,Plataforma,2,3,16,16,4.600,pontos,NaN,Comparação disponível
4,26.0,0.035,CTR consistente,Plataforma,2,3,14,14,1.893,p.p.,25.161,Comparação disponível
5,26.0,0.035,Engagement Rate,Plataforma,2,3,14,16,0.235,p.p.,10.108,Comparação disponível
6,18.0,-0.035,Klike Score,Plataforma,2,1,16,16,-1.688,pontos,NaN,Comparação disponível
7,18.0,-0.035,CTR consistente,Plataforma,2,1,14,16,-1.193,p.p.,-15.856,Comparação disponível
8,18.0,-0.035,Engagement Rate,Plataforma,2,1,14,14,-0.306,p.p.,-13.149,Comparação disponível



Status das comparações:


,Status,Comparações
0,Comparação disponível,9



Análise contextual de duração concluída.



### 5.6. Avaliação da robustez das associações observadas

Diferenças entre médias podem ser influenciadas pela variabilidade dos dados, especialmente quando os grupos possuem poucas observações.

Para investigar a estabilidade das associações contextuais, será utilizado o método de reamostragem **bootstrap não paramétrico**, com 2.000 reamostragens independentes dentro de cada grupo.

Em cada repetição, serão calculadas as médias dos grupos original e alternativo, seguidas de sua diferença.

A distribuição das diferenças permitirá construir intervalos percentílicos de 95%.

Inicialmente, serão avaliadas:

- Inclusão de rosto humano, principal alteração com previsão positiva no Ridge.
- Aumento da duração de 22 para 31 segundos.
- Redução da duração de 22 para 18 segundos.

A alternativa de 26 segundos não será repetida nesta análise porque pertence à mesma faixa de duração da alternativa de 31 segundos, produzindo uma comparação observacional idêntica.

Serão utilizadas as três métricas: Klike Score, CTR consistente e Engagement Rate.

**Limitações:** os intervalos quantificam a variabilidade amostral das diferenças observadas, condicionadas aos grupos e contextos selecionados. Eles não controlam confundimento, viés de seleção ou múltiplas comparações e não devem ser interpretados como evidência de causalidade.

Esta análise permanecerá exploratória e utilizará exclusivamente os registros de treinamento.


In [20]:

bootstrap_contextual = engine.bootstrap_contextual  # Fonte única: src/recommendations_engine.py

In [21]:

# Cenários selecionados para avaliação exploratória
cenarios_bootstrap = [
    ("has_face", 1),
    ("video_duration_s", 31),
    ("video_duration_s", 18)
]

metricas_bootstrap = [
    "klike_score",
    "ctr_consistente",
    "engagement_rate"
]

resultados_bootstrap = []

for feature, valor_proposto in cenarios_bootstrap:
    for metrica in metricas_bootstrap:

        resultado = bootstrap_contextual(
            campanha=campanha_exemplo,
            feature=feature,
            valor_proposto=valor_proposto,
            metrica=metrica,
            n_bootstrap=2000,
            random_state=RANDOM_STATE
        )

        resultados_bootstrap.append(resultado)

tabela_bootstrap = pd.DataFrame(resultados_bootstrap)

print("Campanha:", campanha_exemplo["campaign_id"])

print("\nIntervalos bootstrap para diferenças observacionais:")
display(tabela_bootstrap.round(3))

print("\nQuantidade de comparações realizadas:")
print(len(tabela_bootstrap))

print("\nIntervalos que incluem zero:")
display(
    tabela_bootstrap["Inclui zero"]
    .value_counts()
    .rename_axis("Inclui zero")
    .reset_index(name="Comparações")
)

assert len(tabela_bootstrap) == 9

assert (
    tabela_bootstrap["Status"] == "Bootstrap concluído"
).all()

print("\nAvaliação de robustez concluída.")


Campanha: KLK-0250

Intervalos bootstrap para diferenças observacionais:


,Feature,Proposto,Métrica,Contexto,N original,N proposto,Diferença observada,IC 95% inferior,IC 95% superior,Inclui zero,Unidade,Status
0,has_face,1,Klike Score,Plataforma,30,40,13.562,6.447,21.270,False,pontos,Bootstrap concluído
1,has_face,1,CTR consistente,Plataforma,27,35,2.570,0.251,4.889,False,p.p.,Bootstrap concluído
2,has_face,1,Engagement Rate,Plataforma,29,37,0.635,-0.149,1.501,True,p.p.,Bootstrap concluído
3,video_duration_s,31,Klike Score,Plataforma,16,16,4.600,-6.244,15.457,True,pontos,Bootstrap concluído
4,video_duration_s,31,CTR consistente,Plataforma,14,14,1.893,-1.890,5.439,True,p.p.,Bootstrap concluído
5,video_duration_s,31,Engagement Rate,Plataforma,14,16,0.235,-1.281,1.789,True,p.p.,Bootstrap concluído
6,video_duration_s,18,Klike Score,Plataforma,16,16,-1.688,-14.857,10.382,True,pontos,Bootstrap concluído
7,video_duration_s,18,CTR consistente,Plataforma,14,16,-1.193,-5.015,2.573,True,p.p.,Bootstrap concluído
8,video_duration_s,18,Engagement Rate,Plataforma,14,14,-0.306,-1.701,1.057,True,p.p.,Bootstrap concluído



Quantidade de comparações realizadas:
9

Intervalos que incluem zero:


,Inclui zero,Comparações
0,True,7
1,False,2



Avaliação de robustez concluída.



---

## 6. Seleção e priorização das recomendações

### 6.1. Avaliação sistemática dos candidatos

Antes de selecionar as recomendações finais, será realizada uma avaliação de robustez de todos os cenários alternativos gerados pelo engine.

Para cada cenário, serão reunidos:

- A diferença prevista pelo Ridge reduzido.
- A diferença observacional no Klike Score.
- O intervalo bootstrap de 95% da diferença observacional.
- As evidências complementares de CTR e Engagement Rate.
- O contexto utilizado e a quantidade de observações dos grupos.

Cenários numéricos que representem a mesma comparação entre faixas de duração serão agrupados, preservando a alternativa com maior diferença prevista pelo Ridge. Essa escolha será uma regra de representação, não uma evidência de que a duração selecionada seja ideal.

A interpretação dos intervalos será descritiva:

- **Associação positiva:** intervalo inteiramente acima de zero.
- **Associação negativa:** intervalo inteiramente abaixo de zero.
- **Inconclusiva:** intervalo que inclui zero.
- **Evidência insuficiente:** comparação sem suporte amostral mínimo.

Essas classificações não representam testes causais nem correções para múltiplas comparações.

Nesta etapa, as classificações servirão para caracterizar a evidência disponível. A definição das regras de elegibilidade e priorização será realizada posteriormente.


In [22]:

avaliar_robustez_cenarios = engine.avaliar_robustez_cenarios  # Fonte única: src/recommendations_engine.py

In [23]:

resumo_robustez, detalhes_robustez = (
    avaliar_robustez_cenarios(
        campanha=campanha_exemplo,
        n_bootstrap=2000
    )
)

print("Campanha:", campanha_exemplo["campaign_id"])

print("\nQuantidade de cenários após remover redundâncias:")
print(len(resumo_robustez))

print("\nResumo da robustez dos candidatos:")
display(resumo_robustez.round(3))

print("\nClassificação das associações:")
display(
    resumo_robustez["Classificação"]
    .value_counts()
    .rename_axis("Classificação")
    .reset_index(name="Cenários")
)

print("\nEvidências complementares de CTR e Engagement:")
display(
    detalhes_robustez.loc[
        detalhes_robustez["Métrica"].isin([
            "CTR consistente",
            "Engagement Rate"
        ]),
        [
            "Feature",
            "Proposto",
            "Métrica",
            "Diferença observada",
            "IC 95% inferior",
            "IC 95% superior",
            "Status"
        ]
    ].round(3)
)

# Verificações de consistência
assert len(detalhes_robustez) == len(resumo_robustez) * 3

assert resumo_robustez["Feature"].isin(
    features_modificaveis
).all()

assert np.isfinite(
    resumo_robustez["Delta Ridge"].to_numpy(dtype=float)
).all()

print("\nAvaliação sistemática concluída.")


Campanha: KLK-0250

Quantidade de cenários após remover redundâncias:
7

Resumo da robustez dos candidatos:


,Feature,Original,Proposto,Delta Ridge,Delta Klike observado,IC inferior,IC superior,Contexto,N original,N proposto,Classificação
0,has_face,0,1,11.750,13.562,6.447,21.270,Plataforma,30,40,Associação positiva
1,video_duration_s,22.0,31,0.078,4.600,-6.244,15.457,Plataforma,16,16,Inconclusiva
2,video_duration_s,22.0,18,-0.035,-1.688,-14.857,10.382,Plataforma,16,16,Inconclusiva
3,format,vertical,quadrado,-1.326,14.089,-1.148,27.913,Plataforma,10,22,Inconclusiva
4,format,vertical,horizontal,-1.537,16.459,2.761,29.305,Plataforma,10,38,Associação positiva
5,text_density,medium,low,-6.584,-5.090,-13.273,3.026,Plataforma,28,23,Inconclusiva
6,text_density,medium,high,-14.035,-15.875,-25.651,-6.208,Plataforma,28,19,Associação negativa



Classificação das associações:


,Classificação,Cenários
0,Inconclusiva,4
1,Associação positiva,2
2,Associação negativa,1



Evidências complementares de CTR e Engagement:


,Feature,Proposto,Métrica,Diferença observada,IC 95% inferior,IC 95% superior,Status
1,has_face,1,CTR consistente,2.570,0.251,4.889,Bootstrap concluído
2,has_face,1,Engagement Rate,0.635,-0.149,1.501,Bootstrap concluído
4,video_duration_s,31,CTR consistente,1.893,-1.890,5.439,Bootstrap concluído
5,video_duration_s,31,Engagement Rate,0.235,-1.281,1.789,Bootstrap concluído
7,video_duration_s,18,CTR consistente,-1.193,-5.015,2.573,Bootstrap concluído
8,video_duration_s,18,Engagement Rate,-0.306,-1.701,1.057,Bootstrap concluído
10,format,quadrado,CTR consistente,NaN,NaN,NaN,Amostra insuficiente
11,format,quadrado,Engagement Rate,-0.151,-2.261,1.579,Bootstrap concluído
13,format,horizontal,CTR consistente,NaN,NaN,NaN,Amostra insuficiente
14,format,horizontal,Engagement Rate,-0.140,-2.199,1.403,Bootstrap concluído



Avaliação sistemática concluída.



### 6.2. Definição dos critérios de elegibilidade e priorização

A seleção das recomendações será realizada a partir de regras explícitas, utilizando os resultados preditivos e observacionais calculados anteriormente.

Os critérios adotados serão:

1. **Diferença preditiva positiva:** o cenário deve apresentar Klike Score previsto superior ao da configuração original.
2. **Relevância mínima:** diferenças previstas inferiores a 1 ponto serão classificadas como marginais, evitando priorizar alterações com ganhos numéricos muito pequenos.
3. **Evidência contextual favorável:** para ser elegível como recomendação, o cenário deverá apresentar associação positiva no Klike Score observacional, com intervalo bootstrap inteiramente acima de zero.
4. **Consistência das métricas:** associações negativas com CTR ou Engagement Rate, quando seus intervalos estiverem inteiramente abaixo de zero, impedirão a classificação automática como recomendação elegível.
5. **Ausência de suporte:** comparações inconclusivas poderão ser apresentadas como hipóteses para experimentação, mas não como melhorias sustentadas pelas evidências.

O limite mínimo de 1 ponto é uma **heurística operacional de relevância**, não um limiar estatístico validado. Ele será utilizado para evitar diferenças praticamente nulas, sem representar uma estimativa de incerteza do Ridge.

Os candidatos elegíveis serão ordenados pela diferença prevista no Klike Score. Quando houver várias alternativas para a mesma feature, será preservada somente a alternativa elegível com maior diferença prevista.

A classificação será exploratória, pois as associações não foram ajustadas para confundimento ou múltiplas comparações.

O sistema não criará recomendações artificiais para atingir um número fixo. Quando houver menos de três oportunidades elegíveis, essa limitação será explicitamente apresentada.


In [24]:

priorizar_candidatos = engine.priorizar_candidatos  # Fonte única: src/recommendations_engine.py

In [25]:

avaliacao_candidatos, candidatos_priorizados = (
    priorizar_candidatos(
        resumo_robustez=resumo_robustez,
        detalhes_robustez=detalhes_robustez,
        delta_minimo=1.0,
        max_recomendacoes=3
    )
)

print("Campanha:", campanha_exemplo["campaign_id"])

print("\nClassificação de todos os candidatos:")
display(
    avaliacao_candidatos[
        [
            "Feature",
            "Original",
            "Proposto",
            "Delta Ridge",
            "Classificação",
            "Status decisão",
            "Justificativa"
        ]
    ].round(3)
)

print("\nDistribuição das decisões:")
display(
    avaliacao_candidatos["Status decisão"]
    .value_counts()
    .rename_axis("Decisão")
    .reset_index(name="Quantidade")
)

print("\nCandidatos elegíveis e priorizados:")
display(
    candidatos_priorizados[
        [
            "Prioridade",
            "Feature",
            "Original",
            "Proposto",
            "Delta Ridge",
            "Contexto",
            "Classificação"
        ]
    ].round(3)
)

# Verificações
assert len(candidatos_priorizados) <= 3

assert candidatos_priorizados["Feature"].is_unique

assert (
    candidatos_priorizados["Delta Ridge"] >= 1.0
).all()

assert (
    candidatos_priorizados["Classificação"]
    == "Associação positiva"
).all()

print("\nPriorização executada com sucesso.")


Campanha: KLK-0250

Classificação de todos os candidatos:


,Feature,Original,Proposto,Delta Ridge,Classificação,Status decisão,Justificativa
0,has_face,0,1,11.750,Associação positiva,Elegível,Delta previsto positivo e associação observaci...
1,video_duration_s,22.0,31,0.078,Inconclusiva,Não priorizar,Diferença prevista marginal.
2,video_duration_s,22.0,18,-0.035,Inconclusiva,Não priorizar,Diferença prevista não positiva.
3,format,vertical,quadrado,-1.326,Inconclusiva,Não priorizar,Diferença prevista não positiva.
4,format,vertical,horizontal,-1.537,Associação positiva,Não priorizar,Diferença prevista não positiva.
5,text_density,medium,low,-6.584,Inconclusiva,Não priorizar,Diferença prevista não positiva.
6,text_density,medium,high,-14.035,Associação negativa,Não priorizar,Diferença prevista não positiva.



Distribuição das decisões:


,Decisão,Quantidade
0,Não priorizar,6
1,Elegível,1



Candidatos elegíveis e priorizados:


,Prioridade,Feature,Original,Proposto,Delta Ridge,Contexto,Classificação
0,1,has_face,0,1,11.75,Plataforma,Associação positiva



Priorização executada com sucesso.



---

## 7. Recommendations Engine — Protótipo funcional

### 7.1. Integração dos componentes

O Recommendations Engine será implementado como uma função Python que recebe os atributos de uma campanha e executa automaticamente as seguintes operações:

1. Validar e preparar a campanha utilizando as features do Ridge reduzido.
2. Gerar cenários alternativos modificando uma característica criativa por vez.
3. Calcular as diferenças previstas no Klike Score.
4. Avaliar as associações contextuais utilizando somente os dados de treinamento.
5. Investigar a robustez observacional por bootstrap.
6. Classificar os candidatos e priorizar as alterações elegíveis.
7. Produzir recomendações específicas, quantificadas e compreensíveis para profissionais de marketing.

### Estrutura do output

O sistema retornará:

- Identificador da campanha, quando disponível.
- Klike Score previsto para sua configuração original.
- Quantidade de cenários avaliados.
- Recomendações elegíveis, ordenadas pela diferença prevista no Klike Score.
- Evidências observacionais, contexto utilizado e intervalos bootstrap.
- Diagnóstico das alterações que não atenderam aos critérios.

A função aceitará campanhas existentes e novas entradas compatíveis com o modelo.

**Limitação metodológica:** diferenças previstas entre configurações não constituem efeitos causais. As associações observacionais serão apresentadas separadamente e não serão interpretadas como garantias de melhoria.

Quando houver menos de três oportunidades elegíveis, o sistema informará essa limitação em vez de gerar estimativas artificiais.


In [26]:

descrever_acao = engine.descrever_acao  # Fonte única: src/recommendations_engine.py
recomendar_campanha = engine.recomendar_campanha  # Fonte única: src/recommendations_engine.py

In [27]:

# Executar o Recommendations Engine
resultado_engine = recomendar_campanha(
    campanha=campanha_exemplo,
    max_recomendacoes=3,
    delta_minimo=1.0,
    n_bootstrap=2000
)

print("=" * 65)
print("KLIKE — RECOMMENDATIONS ENGINE")
print("=" * 65)

print("\nCampanha:", resultado_engine["campaign_id"])

print(
    "Klike Score previsto:",
    f"{resultado_engine['score_original']:.3f}"
)

print(
    "Cenários avaliados:",
    resultado_engine["total_cenarios"]
)

print(
    "Recomendações elegíveis:",
    resultado_engine["total_recomendacoes"]
)

print("\nRECOMENDAÇÕES")
print("-" * 65)

if resultado_engine["recomendacoes"]:

    for recomendacao in resultado_engine["recomendacoes"]:

        print(
            f"\n{recomendacao['prioridade']}ª RECOMENDAÇÃO"
        )

        print(recomendacao["texto"])

        print(
            "Amostra contextual:",
            f"N={recomendacao['n_original']} vs "
            f"N={recomendacao['n_proposto']}"
        )

else:
    print(
        "Nenhuma alteração atendeu aos critérios "
        "de elegibilidade estabelecidos."
    )

print("\nDIAGNÓSTICO")
print("-" * 65)

display(
    resultado_engine["avaliacao"][
        [
            "Feature",
            "Original",
            "Proposto",
            "Delta Ridge",
            "Status decisão"
        ]
    ].round(3)
)

# Verificações
assert resultado_engine["total_recomendacoes"] <= 3

assert resultado_engine["total_cenarios"] >= (
    resultado_engine["total_recomendacoes"]
)

assert all(
    recomendacao["delta_ridge"] >= 1.0
    for recomendacao in resultado_engine["recomendacoes"]
)

print("\nExecução concluída.")


KLIKE — RECOMMENDATIONS ENGINE

Campanha: KLK-0250
Klike Score previsto: 66.506
Cenários avaliados: 7
Recomendações elegíveis: 1

RECOMENDAÇÕES
-----------------------------------------------------------------

1ª RECOMENDAÇÃO
Incluir rosto humano no criativo. O Ridge prevê uma diferença de +11.75 pontos no Klike Score para essa configuração alternativa. Nas campanhas comparáveis de LinkedIn, a diferença média observada entre os grupos foi de +13.56 pontos (IC bootstrap 95%: +6.45 a +21.27). Esses resultados não demonstram uma relação causal.
Amostra contextual: N=30 vs N=40

DIAGNÓSTICO
-----------------------------------------------------------------


,Feature,Original,Proposto,Delta Ridge,Status decisão
0,has_face,0,1,11.750,Elegível
1,video_duration_s,22.0,31,0.078,Não priorizar
2,video_duration_s,22.0,18,-0.035,Não priorizar
3,format,vertical,quadrado,-1.326,Não priorizar
4,format,vertical,horizontal,-1.537,Não priorizar
5,text_density,medium,low,-6.584,Não priorizar
6,text_density,medium,high,-14.035,Não priorizar



Execução concluída.



---

## 8. Validação funcional do Recommendations Engine

### 8.1. Teste com campanha informada manualmente

Para verificar se o protótipo funciona independentemente das campanhas existentes no dataset, será criada uma entrada manual utilizando um dicionário Python.

A campanha de demonstração utilizará valores compatíveis com as features do Ridge reduzido, mas não representará uma observação adicional do conjunto de treinamento.

O teste verificará:

- Aceitação de uma campanha fornecida manualmente.
- Validação e preparação das características.
- Execução completa do Recommendations Engine.
- Geração de recomendações quantificadas, quando houver candidatos elegíveis.
- Apresentação do diagnóstico das alternativas avaliadas.

As evidências estatísticas continuarão sendo calculadas exclusivamente com os registros originais de treinamento.

A campanha hipotética não será adicionada à base de referência nem utilizada para ajustar o modelo ou os critérios de seleção.


In [28]:

# Campanha hipotética para testar o input manual
campanha_manual = {
    "campaign_id": "DEMO-MANUAL-001",
    "platform": "Meta",
    "category": "E-commerce",
    "is_retargeting": False,
    "video_duration_s": 16,
    "format": "vertical",
    "has_subtitle": False,
    "has_cta": False,
    "has_hook": False,
    "has_face": False,
    "text_density": "low"
}

# Validar entrada
entrada_manual = preparar_campanha(campanha_manual)

print("Input manual validado:")
display(entrada_manual)

# Executar o Recommendations Engine
resultado_manual = recomendar_campanha(
    campanha=campanha_manual,
    max_recomendacoes=3,
    delta_minimo=1.0,
    n_bootstrap=2000
)

print("\n" + "=" * 65)
print("KLIKE — TESTE COM INPUT MANUAL")
print("=" * 65)

print("Campanha:", resultado_manual["campaign_id"])

print(
    "Klike Score previsto:",
    f"{resultado_manual['score_original']:.3f}"
)

print(
    "Cenários avaliados:",
    resultado_manual["total_cenarios"]
)

print(
    "Recomendações elegíveis:",
    resultado_manual["total_recomendacoes"]
)

print("\nRECOMENDAÇÕES")
print("-" * 65)

if resultado_manual["recomendacoes"]:

    for rec in resultado_manual["recomendacoes"]:
        print(f"\n{rec['prioridade']}ª RECOMENDAÇÃO")
        print(rec["texto"])

        print(
            f"Amostra contextual: "
            f"N={rec['n_original']} vs "
            f"N={rec['n_proposto']}"
        )

else:
    print(
        "Nenhuma alteração atendeu aos critérios "
        "de elegibilidade."
    )

print("\nDIAGNÓSTICO DOS CANDIDATOS")
print("-" * 65)

display(
    resultado_manual["avaliacao"][
        [
            "Feature",
            "Original",
            "Proposto",
            "Delta Ridge",
            "Classificação",
            "Status decisão"
        ]
    ].round(3)
)

print("\nDistribuição das decisões:")
display(
    resultado_manual["avaliacao"]["Status decisão"]
    .value_counts()
    .rename_axis("Decisão")
    .reset_index(name="Quantidade")
)

print("\nTeste manual concluído.")


Input manual validado:


,platform,category,is_retargeting,video_duration_s,format,has_subtitle,has_cta,has_hook,has_face,text_density
0,Meta,E-commerce,0,16.0,vertical,0,0,0,0,low



KLIKE — TESTE COM INPUT MANUAL
Campanha: DEMO-MANUAL-001
Klike Score previsto: 34.258
Cenários avaliados: 10
Recomendações elegíveis: 3

RECOMENDAÇÕES
-----------------------------------------------------------------

1ª RECOMENDAÇÃO
Adicionar um hook nos primeiros 3 segundos. O Ridge prevê uma diferença de +17.75 pontos no Klike Score para essa configuração alternativa. Nas campanhas comparáveis de Meta / E-commerce, a diferença média observada entre os grupos foi de +18.33 pontos (IC bootstrap 95%: +13.58 a +23.45). Esses resultados não demonstram uma relação causal.
Amostra contextual: N=31 vs N=30

2ª RECOMENDAÇÃO
Incluir rosto humano no criativo. O Ridge prevê uma diferença de +11.75 pontos no Klike Score para essa configuração alternativa. Nas campanhas comparáveis de Meta / E-commerce, a diferença média observada entre os grupos foi de +9.95 pontos (IC bootstrap 95%: +3.72 a +16.34). Esses resultados não demonstram uma relação causal.
Amostra contextual: N=36 vs N=25

3ª RECOME

,Feature,Original,Proposto,Delta Ridge,Classificação,Status decisão
0,has_hook,0,1,17.753,Associação positiva,Elegível
1,has_face,0,1,11.750,Associação positiva,Elegível
2,has_cta,0,1,7.693,Inconclusiva,Teste exploratório
3,has_subtitle,0,1,7.386,Associação positiva,Elegível
4,text_density,low,medium,6.584,Inconclusiva,Teste exploratório
5,video_duration_s,16.0,26,0.087,Inconclusiva,Não priorizar
6,video_duration_s,16.0,11,-0.043,Inconclusiva,Não priorizar
7,format,vertical,quadrado,-1.326,Inconclusiva,Não priorizar
8,format,vertical,horizontal,-1.537,Inconclusiva,Não priorizar
9,text_density,low,high,-7.451,Inconclusiva,Não priorizar



Distribuição das decisões:


,Decisão,Quantidade
0,Não priorizar,5
1,Elegível,3
2,Teste exploratório,2



Teste manual concluído.


In [29]:

# Verificar se a campanha informada não foi modificada
assert campanha_manual == {
    "campaign_id": "DEMO-MANUAL-001",
    "platform": "Meta",
    "category": "E-commerce",
    "is_retargeting": False,
    "video_duration_s": 16,
    "format": "vertical",
    "has_subtitle": False,
    "has_cta": False,
    "has_hook": False,
    "has_face": False,
    "text_density": "low"
}

# Verificar a estrutura do retorno
campos_esperados = {
    "campaign_id",
    "score_original",
    "total_cenarios",
    "total_recomendacoes",
    "recomendacoes",
    "avaliacao",
    "evidencias"
}

assert campos_esperados.issubset(resultado_manual.keys())

# Conferir a quantidade de recomendações
assert resultado_manual["total_recomendacoes"] == len(
    resultado_manual["recomendacoes"]
)

assert 0 <= resultado_manual["total_recomendacoes"] <= 3

# Verificar o score previsto
assert np.isfinite(resultado_manual["score_original"])

# Confirmar que as recomendações são elegíveis
assert all(
    rec["delta_ridge"] >= 1.0
    for rec in resultado_manual["recomendacoes"]
)

# Confirmar que a campanha manual não foi adicionada ao treino
assert "DEMO-MANUAL-001" not in set(
    df_evidencias["campaign_id"]
)

print("Todas as verificações de integridade passaram.")


Todas as verificações de integridade passaram.



### 8.2. Testes de integridade, entradas inválidas e data leakage

Após confirmar o funcionamento do Recommendations Engine com uma campanha manual, serão realizados testes adicionais para verificar a robustez da interface de entrada.

Os testes contemplarão:

1. Rejeição de campanhas com características obrigatórias ausentes.
2. Rejeição de categorias e valores booleanos inválidos.
3. Rejeição de durações negativas.
4. Verificação de que métricas de performance, como `ctr` e `klike_score`, não são utilizadas como features preditivas.
5. Verificação da quantidade e da ordenação das recomendações retornadas.

A separação entre características disponíveis antes da veiculação e métricas observadas posteriormente é fundamental para reduzir o risco de data leakage.

Esses testes verificam propriedades funcionais do protótipo. Eles não substituem uma avaliação de generalização do modelo em dados externos ou a validação causal das recomendações.


In [30]:

# Casos inválidos construídos a partir do input manual
casos_invalidos = {
    "Feature obrigatória ausente": {
        k: v for k, v in campanha_manual.items()
        if k != "has_cta"
    },
    "Plataforma desconhecida": {
        **campanha_manual,
        "platform": "Plataforma inexistente"
    },
    "Formato desconhecido": {
        **campanha_manual,
        "format": "circular"
    },
    "Duração negativa": {
        **campanha_manual,
        "video_duration_s": -10
    },
    "Booleano inválido": {
        **campanha_manual,
        "has_hook": "false"
    },
    "Booleano fora do domínio": {
        **campanha_manual,
        "has_face": 2
    }
}

resultados_validacao = []

for nome, campanha_invalida in casos_invalidos.items():
    try:
        preparar_campanha(campanha_invalida)

    except (ValueError, TypeError) as erro:
        resultados_validacao.append({
            "Teste": nome,
            "Resultado": "Rejeitado corretamente",
            "Motivo": str(erro)
        })

    else:
        resultados_validacao.append({
            "Teste": nome,
            "Resultado": "Falha",
            "Motivo": "Input inválido foi aceito."
        })

tabela_validacao = pd.DataFrame(resultados_validacao)

display(tabela_validacao)

assert (
    tabela_validacao["Resultado"] == "Rejeitado corretamente"
).all()

print("\nTodos os inputs inválidos foram rejeitados.")


,Teste,Resultado,Motivo
0,Feature obrigatória ausente,Rejeitado corretamente,Features ausentes: ['has_cta']
1,Plataforma desconhecida,Rejeitado corretamente,Valor inválido para platform: Plataforma inexi...
2,Formato desconhecido,Rejeitado corretamente,Valor inválido para format: circular. Valores ...
3,Duração negativa,Rejeitado corretamente,video_duration_s deve ser um número positivo e...
4,Booleano inválido,Rejeitado corretamente,"has_hook deve ser True, False, 0 ou 1."
5,Booleano fora do domínio,Rejeitado corretamente,"has_face deve ser True, False, 0 ou 1."



Todos os inputs inválidos foram rejeitados.


In [31]:

# Uma campanha com métricas posteriores à veiculação
campanha_com_metricas = {
    **campanha_manual,
    "ctr": 0.99,
    "roas": 99.0,
    "klike_score": 100,
    "revenue": 1000000
}

# O input do Ridge deve continuar idêntico
entrada_original = preparar_campanha(campanha_manual)
entrada_com_metricas = preparar_campanha(
    campanha_com_metricas
)

pd.testing.assert_frame_equal(
    entrada_original,
    entrada_com_metricas
)

# As previsões devem ser iguais
previsao_original = modelo_ridge.predict(
    entrada_original
)

previsao_com_metricas = modelo_ridge.predict(
    entrada_com_metricas
)

assert np.allclose(
    previsao_original,
    previsao_com_metricas
)

print("Métricas pós-veiculação ignoradas corretamente.")
print("Previsões idênticas:", True)

# Conferir as três recomendações do teste manual
recomendacoes = resultado_manual["recomendacoes"]

assert len(recomendacoes) == 3

prioridades = [
    rec["prioridade"]
    for rec in recomendacoes
]

assert prioridades == [1, 2, 3]

deltas = [
    rec["delta_ridge"]
    for rec in recomendacoes
]

assert deltas == sorted(deltas, reverse=True)

features_recomendadas = [
    rec["feature"]
    for rec in recomendacoes
]

assert len(features_recomendadas) == len(
    set(features_recomendadas)
)

# Todas as recomendações devem ter suporte contextual
assert all(
    rec["n_original"] >= 10
    and rec["n_proposto"] >= 10
    for rec in recomendacoes
)

print("\nQuantidade de recomendações:", len(recomendacoes))
print("Features recomendadas:", features_recomendadas)
print("Deltas ordenados:", [round(d, 3) for d in deltas])

print("\nVerificações de integridade concluídas.")


Métricas pós-veiculação ignoradas corretamente.
Previsões idênticas: True

Quantidade de recomendações: 3
Features recomendadas: ['has_hook', 'has_face', 'has_subtitle']
Deltas ordenados: [17.753, 11.75, 7.386]

Verificações de integridade concluídas.



---

## 9. Verificação final e reprodutibilidade

### 9.1. Auditoria da implementação

A etapa final tem como objetivo verificar se o Recommendations Engine produz resultados consistentes em execuções repetidas e se preserva os critérios metodológicos definidos.

Serão verificadas:

- A integridade da divisão entre treinamento e teste.
- A utilização exclusiva das features selecionadas na Parte 2.
- A disponibilidade dos arquivos necessários para reproduzir as previsões.
- A estabilidade das recomendações em execuções repetidas com a mesma configuração.
- O atendimento ao requisito de três recomendações na campanha de demonstração.

O bootstrap utiliza uma semente aleatória fixa para permitir a reprodução dos resultados.

Essas verificações não substituem validação externa, experimentos controlados ou testes de causalidade.

Após sua execução, o notebook deverá ser reiniciado e executado integralmente, do início ao fim, para confirmar que não depende de variáveis mantidas em memória por execuções anteriores.


In [32]:

# 1. Validar arquivos e separação dos dados
assert all(caminho.is_file() for caminho in arquivos.values())

assert set(X_train.index).isdisjoint(set(X_test.index))
assert set(df_evidencias.index) == set(X_train.index)

# 2. Garantir a ausência de métricas pós-veiculação nas features
features_proibidas = {
    "klike_score",
    "ctr",
    "roas",
    "engagement_rate",
    "revenue",
    "conversions",
    "clicks",
    "impressions",
    "spend",
    "cpc"
}

assert features_proibidas.isdisjoint(features_reduzidas)

# Verificar as features esperadas pelo pipeline, quando disponíveis
if hasattr(modelo_ridge, "feature_names_in_"):
    assert list(modelo_ridge.feature_names_in_) == features_reduzidas

# 3. Reexecutar o engine com o mesmo input
resultado_repetido = recomendar_campanha(
    campanha=campanha_manual,
    max_recomendacoes=3,
    delta_minimo=1.0,
    n_bootstrap=2000
)

# 4. Comparar resultados das execuções
assert np.isclose(
    resultado_manual["score_original"],
    resultado_repetido["score_original"]
)

assert (
    resultado_manual["total_recomendacoes"]
    == resultado_repetido["total_recomendacoes"]
)

for rec_original, rec_repetida in zip(
    resultado_manual["recomendacoes"],
    resultado_repetido["recomendacoes"]
):
    assert rec_original["feature"] == rec_repetida["feature"]

    assert np.isclose(
        rec_original["delta_ridge"],
        rec_repetida["delta_ridge"]
    )

    assert np.isclose(
        rec_original["ic_inferior"],
        rec_repetida["ic_inferior"]
    )

    assert np.isclose(
        rec_original["ic_superior"],
        rec_repetida["ic_superior"]
    )

# 5. Requisito de demonstração
assert resultado_repetido["total_recomendacoes"] == 3

print("AUDITORIA FINAL")
print("-" * 50)
print("Arquivos necessários: OK")
print("Isolamento treino/teste: OK")
print("Features preditivas: OK")
print("Resultados reproduzíveis: OK")
print("Três recomendações na demonstração: OK")
print("\nAuditoria concluída.")


AUDITORIA FINAL
--------------------------------------------------
Arquivos necessários: OK
Isolamento treino/teste: OK
Features preditivas: OK
Resultados reproduzíveis: OK
Três recomendações na demonstração: OK

Auditoria concluída.



---

## 10. Conclusões e limitações

### 10.1. Resultados do Recommendations Engine

Foi desenvolvido um protótipo funcional de recomendações para criativos publicitários, combinando o Ridge reduzido da Parte 2 com associações estatísticas contextualizadas pelos dados de treinamento.

O sistema é capaz de receber uma campanha individual, validar seus atributos, gerar configurações alternativas, calcular diferenças previstas no Klike Score e selecionar recomendações fundamentadas em evidências observacionais.

Na demonstração com a campanha hipotética `DEMO-MANUAL-001`, foram avaliados 10 cenários alternativos e selecionadas três recomendações:

| Prioridade | Recomendação | Diferença prevista no Klike Score |
|---|---|---|
| 1 | Adicionar hook nos primeiros 3 segundos | +17,75 pontos |
| 2 | Incluir rosto humano no criativo | +11,75 pontos |
| 3 | Adicionar legendas ao vídeo | +7,39 pontos |

As três alterações apresentaram associações observacionais positivas com o Klike Score em campanhas comparáveis do Meta, na categoria E-commerce.

As recomendações foram priorizadas pela diferença prevista no Ridge, sem combinar artificialmente métricas com escalas distintas.

### 10.2. Limitações metodológicas

Apesar dos resultados, algumas limitações devem ser consideradas:

**Ausência de identificação causal:** as diferenças previstas representam cenários alternativos avaliados pelo modelo, enquanto as diferenças observacionais representam associações entre campanhas distintas. Nenhuma delas demonstra que uma alteração causará o ganho apresentado.

**Modelo linear aditivo:** o Ridge reduzido não possui interações explícitas entre plataformas e características criativas. A contextualização das recomendações depende, portanto, principalmente da camada estatística observacional.

**Amostra limitada:** o dataset contém 500 campanhas, sendo 400 utilizadas no treinamento. Segmentações específicas podem apresentar suporte amostral insuficiente.

**Incerteza estatística:** os intervalos bootstrap quantificam a variabilidade das diferenças observacionais, mas não controlam confundimento, múltiplas comparações ou a incerteza das previsões do Ridge. A seleção dos candidatos também pode introduzir viés de seleção nas estimativas apresentadas.

**Critérios heurísticos:** o mínimo de 10 observações por grupo e o delta mínimo de 1 ponto são regras operacionais exploratórias, não limites estatisticamente otimizados.

**Disponibilidade das recomendações:** nem toda campanha apresenta três alterações elegíveis. O sistema pode retornar menos sugestões quando as evidências disponíveis não sustentam outras oportunidades.

**Escopo das simulações:** cada alteração é avaliada isoladamente. A implementação não estima os efeitos de combinar várias modificações simultaneamente.

### 10.3. Considerações finais

O Recommendations Engine demonstra como resultados exploratórios e preditivos podem ser transformados em sugestões acionáveis para uma aplicação de marketing.

A abordagem híbrida permite apresentar tanto a diferença prevista pelo modelo quanto o contexto estatístico que fundamenta cada recomendação, mantendo essas duas fontes de informação conceitualmente separadas.

O protótipo atende aos requisitos funcionais demonstrados, incluindo entrada manual, recomendações quantificadas, contextualização e priorização.

Como evolução, seria importante validar as recomendações com experimentos controlados, avaliar sua estabilidade em novas campanhas e investigar modelos capazes de representar interações entre características dos criativos e contexto de veiculação.
